In [30]:
import json
import pandas as pd
from tqdm import tqdm
import os
import re

In [31]:
def create_df(content, label):
    data = content['data']

    rows = []

    for window in data:
        window_data = {}
        for idx, capture in enumerate(window[0:25]):
            # Check if capture contains all expected keys
            timestamp = capture.get('timestamp', None)  # Default to None if key is missing
            flex_data = capture.get('flex_data', [])
            accelerometer = capture.get('accelerometer', {})
            gyroscope = capture.get('gyroscope', {})

            # Add timestamp to the row
            window_data[f'time_stamp_{idx}'] = timestamp

            # Add flex data to the row
            for idxx, finger in enumerate(flex_data):
                window_data[f'flex_finger_{idx}_{idxx}'] = finger

            # Add accelerometer data to the row
            for value in ['x', 'y', 'z']:  # Ensure consistent keys for accelerometer
                window_data[f'acc_{idx}_{value}'] = accelerometer.get(value, None)

            # Add gyroscope data to the row
            for value in ['x', 'y', 'z']:  # Ensure consistent keys for gyroscope
                window_data[f'gyro_{idx}_{value}'] = gyroscope.get(value, None)

        rows.append(window_data)

    # Create DataFrame from rows
    df = pd.DataFrame(rows)

    # Add the label column
    df['label'] = label
    
    return df

In [32]:
root_dir = 'dataset/json/'

all_dataframes = []

for data_json in tqdm(os.listdir(root_dir)):
    if data_json.endswith('json'):
        with open(os.path.join(root_dir, data_json)) as file:
            content = json.loads(file.read())
            
        label = data_json.split('.')[0]
        label = label.split("-")[0]
        label = re.sub(r'\d+', '', label)

        df = create_df(content, label)

        all_dataframes.append(df)
        
        
# Combine all dataframes into one
combined_df = pd.concat(all_dataframes, ignore_index=True)

# Save the combined dataframe to a CSV file
combined_df.to_csv('dataset.csv', index=False)

100%|██████████████████████████████████████████| 52/52 [00:00<00:00, 237.19it/s]


In [33]:
print(combined_df.shape)
print(combined_df['label'].value_counts())

(520, 301)
good_morning      110
how_are_you       110
alsalam_alikum    110
wht_ur_name       100
how_old_are_u      90
Name: label, dtype: int64


In [35]:
combined_df.isnull().sum()

time_stamp_0       0
flex_finger_0_0    0
flex_finger_0_1    0
flex_finger_0_2    0
flex_finger_0_3    0
                  ..
acc_24_z           0
gyro_24_x          0
gyro_24_y          0
gyro_24_z          0
label              0
Length: 301, dtype: int64

<br><br><br><br><br>